# Step 0 - Look inside the Gotham dataset archive

This notebook reads `GothamDataset2025.zip` directly from Google Drive (nothing is unzipped) and reports what is inside: files, columns, labels, time structure, protocol stacks, duplicate packets and label ambiguity.

* **Runtime:** CPU is enough (*Runtime -> Change runtime type -> CPU*). Keep GPU time for training.
* **Time:** inventory 1-3 min; the full scan of ~9 GB of CSV text roughly 30-60 min.
* **Disconnected?** Reconnect and run the cells again. Finished files are skipped; only the file that was being read is redone.
* **Results:** `MyDrive/gotham_ids/reports/explore/` - `SUMMARY.md` is the file to share.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/GothamDataset2025.zip"  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}

In [ ]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

In [ ]:
#@title 3. Self-test on a small synthetic archive (about 30 s)
!python -m gids selftest

In [ ]:
#@title 4. Inventory: list the archive, read the first rows of a few CSVs, measure read speed
!python -u -m gids inventory
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/reports/explore/INVENTORY.md").read()))

In [ ]:
#@title 5. Full scan of every CSV (resumable: after a disconnect just run cells 2 and 5 again)
!python -u -m gids explore

In [ ]:
#@title 6. Summary and charts
from IPython.display import Image, Markdown, display
R = f"{WORK_DIR}/reports/explore"
display(Markdown(open(f"{R}/SUMMARY.md").read()))
for name in ["labels.png", "classes.png", "device_type_by_label.png", "timeline.png"]:
    if os.path.exists(f"{R}/{name}"):
        display(Image(f"{R}/{name}"))

In [ ]:
#@title 7. Plain text of the summary (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/explore/SUMMARY.md").read())

In [ ]:
#@title 8. (optional) Download the whole reports folder as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/explore_reports", "zip", f"{WORK_DIR}/reports/explore")
files.download("/content/explore_reports.zip")

### If something goes wrong
* `status` shows how far each stage got and whether recent runs finished or were killed.
* `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* A killed run leaves its lock behind; it is taken over automatically after 5 minutes (immediately on the same machine). `unlock --yes` removes it by hand.
* If reading the zip from Drive keeps failing, copy it to the Colab disk once (`!cp "$ZIP_PATH" /content/`) and set `ZIP_PATH = "/content/GothamDataset2025.zip"` in cell 1.

In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30